In [2]:
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Naveen\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [5]:
# base_url="/content/drive/MyDrive/Viora/"
base_url="./"

In [7]:
import numpy as np
import pandas as pd
import nltk
from sklearn.preprocessing import OneHotEncoder
def wordTokenize(text:str):
    return nltk.word_tokenize(text)

In [6]:
# Loading Dataset and Preprocessing
data = pd.read_csv(f'{base_url}emotion_sentimen_dataset.csv')
data.head()
df = data["text"].to_frame(name= "text")

def normalizeData(df):
    for index,row in df.iterrows():
        text = row['text']
        text = text.lower()
        text.strip()
        text.strip('''
        !"#$%&'()*+,-./:;<=>?@[]\\^_`{|}~
        ''')
        row['text'] = text
    return df

df = normalizeData(df)

vocabulary = {
    "<PAD>": 0,
    "<UNK>": 1}

currIdx=2

max_len = 60

def buildVocabulary(df):
    global vocabulary
    global currIdx
    global max_len
    for idx,row in df.iterrows():
        text = row['text']
        text = wordTokenize(text)
        numSequence = []

        for word in text:
            if word not in vocabulary:
                vocabulary[word] = currIdx
                numSequence.append(currIdx)
                currIdx+=1
            else:
                numSequence.append(vocabulary[word])

        numSequence =  np.array(numSequence)
        if(len(numSequence)<max_len):
            padLen = max_len-len(numSequence)
            numSequence = np.append(numSequence ,  [vocabulary["<PAD>"]]*padLen)
        else:
            numSequence = numSequence[0:max_len]

        row['text'] = numSequence

# doing One hot encoding of labels
encoder = OneHotEncoder(sparse_output = False , handle_unknown = 'ignore')
emotion_column = data['Emotion'].tolist()
unique_emotion_list = np.unique(emotion_column)
emotion = np.array(emotion_column).reshape(-1,1)

y = encoder.fit_transform(emotion).astype(int)

y_df = pd.DataFrame(y , columns = unique_emotion_list  ) # final One Hot Encoded Labels
buildVocabulary(df)
print("Vocabulary Size : " , currIdx)
df.head(10)

KeyboardInterrupt: 

In [ ]:
print(len(df))
print(df.head())
print(y_df.head())

839555
                                                text
0  [2, 3, 4, 5, 6, 7, 8, 9, 10, 2, 11, 12, 7, 13,...
1  [15, 16, 17, 18, 19, 2, 11, 20, 0, 0, 0, 0, 0,...
2  [2, 21, 22, 7, 23, 2, 24, 7, 25, 26, 27, 28, 2...
3  [41, 42, 43, 44, 45, 46, 29, 2, 11, 47, 48, 49...
4  [2, 11, 56, 57, 58, 59, 60, 5, 61, 47, 35, 62,...
   anger  boredom  empty  enthusiasm  fun  happiness  hate  love  neutral  \
0      0        0      0           0    0          0     1     0        0   
1      0        0      0           0    0          0     0     0        1   
2      0        0      0           0    0          0     0     0        1   
3      1        0      0           0    0          0     0     0        0   
4      0        0      0           0    0          0     0     0        1   

   relief  sadness  surprise  worry  
0       0        0         0      0  
1       0        0         0      0  
2       0        0         0      0  
3       0        0         0      0  
4       0        0    

In [ ]:
from sklearn.model_selection import train_test_split

# X = df.drop(columns = 'emotion_label'  ) # it will be treated as object because pandas will store this column as object even though this contains a sequence of int 64  (the column itself it object type ) and keras required integer type
# print(X)
# X = np.array(df['text'].tolist() , dtype = np.int32) # because keras required the input to be a 2d or 3d tensor and drop column
X = np.stack(df['text'].values).astype(np.int32)
X_train , X_test , y_train , y_test = train_test_split(X , y_df , test_size =0.2 , random_state = 42)

print("success")

success


In [ ]:
# Training
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding , GlobalAveragePooling1D , Dense

num_classes = 13 # unique number of classes or len(unique_emotion_list)
vocab_size = len(vocabulary) # actual Vocab Size
embedding_dim = 50
max_len = 10

# Defining Model
model = Sequential([
    Embedding(input_dim = vocab_size,
              output_dim = embedding_dim,
              input_length = max_len),
    GlobalAveragePooling1D(),
    Dense(64 , activation = 'relu'),
    Dense(num_classes , activation = 'softmax')
])

model.compile(
    optimizer = 'adam',
    loss= 'categorical_crossentropy',
    metrics = ['accuracy']
)
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_2      │ ?                      │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:

history = model.fit(
    X_train,
    y_train,
    epochs =10,
    batch_size = 32,
    validation_data = (X_test , y_test)
)

loss , accuracy = model.evaluate(X_test , y_test)
print("Validation Accuracy : " , accuracy)


Epoch 1/10
20989/20989 ━━━━━━━━━━━━━━━━━━━━ 724s 34ms/step - accuracy: 0.8924 - loss: 0.4585 - val_accuracy: 0.9930 - val_loss: 0.0374
Epoch 2/10
20989/20989 ━━━━━━━━━━━━━━━━━━━━ 700s 33ms/step - accuracy: 0.9948 - loss: 0.0286 - val_accuracy: 0.9968 - val_loss: 0.0209
Epoch 3/10
20989/20989 ━━━━━━━━━━━━━━━━━━━━ 747s 34ms/step - accuracy: 0.9974 - loss: 0.0136 - val_accuracy: 0.9977 - val_loss: 0.0113
Epoch 4/10
20989/20989 ━━━━━━━━━━━━━━━━━━━━ 694s 33ms/step - accuracy: 0.9985 - loss: 0.0078 - val_accuracy: 0.9977 - val_loss: 0.0119
Epoch 5/10
20989/20989 ━━━━━━━━━━━━━━━━━━━━ 740s 33ms/step - accuracy: 0.9988 - loss: 0.0060 - val_accuracy: 0.9987 - val_loss: 0.0075
Epoch 6/10
20989/20989 ━━━━━━━━━━━━━━━━━━━━ 751s 33ms/step - accuracy: 0.9990 - loss: 0.0049 - val_accuracy: 0.9989 - val_loss: 0.0063
Epoch 7/10
20989/20989 ━━━━━━━━━━━━━━━━━━━━ 736s 33ms/step - accuracy: 0.9991 - loss: 0.0041 - val_accuracy: 0.9990 - val_loss: 0.0062
Epoch 8/10
20989/20989 ━━━━━━━━━━━━━━━━━━━━ 753s 34ms/s

In [ ]:
# Saving the model and dumping the vocabulary, unique_emotion_list , max_len
import pickle
model.save(f'{base_url}emotion_model.keras')

with open(f'{base_url}vocab.pkl' , 'wb') as f:
  pickle.dump(vocabulary , f)

with open(f'{base_url}unique_emotion_list.pkl' , 'wb') as f:
  pickle.dump(unique_emotion_list , f)

with open(f'{base_url}max_len.pkl' , 'wb') as f:
  pickle.dump(max_len , f)

In [14]:
# Predicting an Emotion Based on Our Trained Model
import pickle
from keras.models import load_model
text = input("Enter Some Text ")

vocabulary = {}
unique_emotion_list = []
max_len = 0
with open(f'{base_url}vocab.pkl' , 'rb') as f:
  vocabulary=pickle.load( f)

with open(f'{base_url}unique_emotion_list.pkl' , 'rb') as f:
  unique_emotion_list=pickle.load(f)

with open(f'{base_url}max_len.pkl' , 'rb') as f:
  max_len=pickle.load(f)

with open(f'{base_url}max_len.pkl' , 'rb') as f:
  max_len=pickle.load(f)
def preProcessInput(text):
  text = text.lower().strip().strip('''
        !"#$%&'()*+,-./:;<=>?@[]\\^_`{|}~
        ''')
  tokens = wordTokenize(text)
  sequence = []
  for word in tokens:
    if word in vocabulary:
      sequence.append(vocabulary[word])
    else:
      sequence.append(vocabulary["<UNK>"])
  if (len(sequence) < max_len):
    padlen = max_len - len(sequence)
    sequence.extend([vocabulary["<PAD>"]]*padlen)
  else:
    sequence = sequence[:max_len]
  return np.array(sequence).reshape(-1 , 1)

processed = preProcessInput(text)
model = load_model(f'{base_url}emotion_model.keras')
prediction = model.predict(processed)

predicted_index = np.argmax(prediction, axis = 1)[0]
predicted_emotion = unique_emotion_list [predicted_index]
print("Predicted Emotion : " , predicted_emotion)


Enter Some Text  From here go there


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 228ms/step
Predicted Emotion :  sadness


In [ ]:
# It Needs Training for Hindi words and Sentences